Author: Alana Pooler
<br>
Purpose: Complete Homework 10

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sqrt, col
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 16:18:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 16:18:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Part 1: Creating Streaming Data Using `rate`

First, we will set up a data stream using the `rate` format.

In [2]:
df = spark \
    .readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

Before starting the stream, we want to do a couple transformations:
* Find the square root of the rate 'value'
* Find mod 4 of the rate 'value'

In [3]:
df_transformed = df.select(
    col("timestamp"),
    col("value"),
    sqrt(col("value")).alias("sqrt_value"),
    (col("value") % 4).alias("mod_4")
)

Now, we can start the stream and apply the transformations. We will use .format("memory") to write the output to memory.

In [8]:
query = df_transformed.writeStream \
    .format("memory") \
    .queryName("rate_table") \
    .outputMode("append") \
    .start()

26/04/21 16:23:23 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-05fba774-2bf9-40f5-a8ac-42ca69d94a9e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:23:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


After about 30 seconds, we will stop the query and look at the results.

In [9]:
query.stop()
spark.sql("SELECT * FROM rate_table").show()

+--------------------+-----+------------------+-----+
|           timestamp|value|        sqrt_value|mod_4|
+--------------------+-----+------------------+-----+
|2026-04-21 16:23:...|    0|               0.0|    0|
|2026-04-21 16:23:...|    1|               1.0|    1|
|2026-04-21 16:23:...|    2|1.4142135623730951|    2|
|2026-04-21 16:23:...|    3|1.7320508075688772|    3|
|2026-04-21 16:23:...|    4|               2.0|    0|
|2026-04-21 16:23:...|    5|  2.23606797749979|    1|
|2026-04-21 16:23:...|    6| 2.449489742783178|    2|
|2026-04-21 16:23:...|    7|2.6457513110645907|    3|
|2026-04-21 16:23:...|    8|2.8284271247461903|    0|
|2026-04-21 16:23:...|    9|               3.0|    1|
|2026-04-21 16:23:...|   10|3.1622776601683795|    2|
|2026-04-21 16:23:...|   11|   3.3166247903554|    3|
|2026-04-21 16:23:...|   12|3.4641016151377544|    0|
|2026-04-21 16:23:...|   13| 3.605551275463989|    1|
|2026-04-21 16:23:...|   14|3.7416573867739413|    2|
|2026-04-21 16:23:...|   15|

26/04/21 16:23:41 WARN DAGScheduler: Failed to cancel job group e11d8208-fa03-4a7d-94e0-77608616c5e0. Cannot find active jobs for it.
26/04/21 16:23:41 WARN DAGScheduler: Failed to cancel job group e11d8208-fa03-4a7d-94e0-77608616c5e0. Cannot find active jobs for it.


#Part 2: Using data from a CSV with a Pipeline

First we need to read in the bikeDetails_for_fit.csv file as a spark data frame.

In [10]:
df = spark.read.csv("bikeDetails_for_fit.csv", header=True, inferSchema=True)

Next, we will do a few transformations on the bike data:
* Log transform and rename selling_price and km_driven columns
* Rename selling_price to 'label'
* Convert 'owner' column to binary

In [13]:
sql_trans = SQLTransformer(
    statement = """
                SELECT
                  log(selling_price) as label,
                  year,
                  log(km_driven) as log_km_driven,
                  CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
                FROM __THIS__
                """
)

Now we will use `VectorAssembler` to create a features column that includes the year, log_km_driven, and one_owner variables.

In [14]:
assembler = VectorAssembler(
    inputCols=["year", "log_km_driven", "one_owner"],
    outputCol="features"
)

Next, we will create a pipeline consisting of the SQLTransformer and VectorAssembler transformations. We will also fit the pipeline to the bike data we read in earlier.

In [15]:
pipeline = Pipeline(stages=[sql_trans, assembler])
pipeline_model = pipeline.fit(df)

Now we will set up a read stream to look for .csv files in a folder and transform them with the pipeline we set up. We will run this to apply the transformations to the 5 other bikeDetails data sets.

In [17]:
# define schema
schema = df.schema

# set up read stream
stream_df = (
    spark.readStream
    .schema(schema)
    .option("header", True)
    .csv("data")
)

# apply transformations using .transform()
transformed_stream = pipeline_model.transform(stream_df)

Now we can write the output to the console using writeStream. 

We will start with the "data" folder empty, and drop each file into the folder one by one. This will apply both of the transformations we added to the pipeline to these 5 files.

In [35]:
query = transformed_stream.writeStream \
        .format("console") \
        .outputMode("append") \
        .start()

26/04/21 16:43:26 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1fe1aaf2-0f7b-45f5-abfe-0307aeaf0fb5. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:43:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

Once all 5 files are done, we can stop the query.

In [36]:
query.stop()

26/04/21 16:43:53 WARN DAGScheduler: Failed to cancel job group 5d3eaeb9-3f03-4dca-bf4f-b78c5bca70a1. Cannot find active jobs for it.
26/04/21 16:43:53 WARN DAGScheduler: Failed to cancel job group 5d3eaeb9-3f03-4dca-bf4f-b78c5bca70a1. Cannot find active jobs for it.
